# Step 1 — Evaluate the base student

This notebook evaluates `Qwen/Qwen3-4B-Thinking-2507` on the three MOPD domains: math (AIME25/26), instruction following (IFBench/IFEval), and software engineering (SWE-bench Verified). It uses the tokenizer's bundled chat template, Unsloth inference, one visible GPU, resumable JSONL outputs, and line-oriented progress suitable for Papermill logs.

Important: SWE-bench's score is only valid after the official Docker harness executes the generated patches. Patch-generation success is never reported as task success.

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
print("HF mirror:", os.environ["HF_ENDPOINT"], flush=True)
print("Physical CUDA device exposed to this kernel:", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)

In [ ]:
# Papermill parameters
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_DIR = "results/student_base_eval"
SEED = 42

LIMIT_AIME25 = None
LIMIT_AIME26 = None
LIMIT_IFBENCH = None
LIMIT_IFEVAL = None
LIMIT_SWE = None

MAX_SEQ_LENGTH = 32768
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0

RUN_MATH = True
RUN_IF = True
RUN_SWE = True
RUN_SWE_HARNESS = True
SWE_MAX_WORKERS = 4
SWE_RUN_ID = "student-base"
RESUME = True
MAX_CONSECUTIVE_ERRORS = 3
INSTALL_DEPS = False

In [ ]:
# Optional dependency bootstrap. For a CUDA server, preinstall a PyTorch/Unsloth build
# matching the server's CUDA driver whenever possible. This cell never upgrades torch.
import importlib.util
import subprocess
import sys

PACKAGE_BY_MODULE = {
    "unsloth": "unsloth",
    "datasets": "datasets",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "ifbench": "git+https://github.com/allenai/IFBench.git@1c40f0c10d9b5c5c2f10a175a28007ebb64f7f4d",
    "papermill": "papermill",
}
if RUN_SWE_HARNESS:
    PACKAGE_BY_MODULE["swebench"] = "swebench==5.0.2"
missing = [package for module, package in PACKAGE_BY_MODULE.items() if importlib.util.find_spec(module) is None]
if missing and INSTALL_DEPS:
    print("Installing missing Python packages: " + ", ".join(missing), flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
elif missing:
    raise ImportError(
        "Missing packages: " + ", ".join(missing) +
        ". Install them in the CUDA environment, or run Papermill with -p INSTALL_DEPS true."
    )
print("Dependency check passed.", flush=True)

## Runtime and model loading

Because `CUDA_VISIBLE_DEVICES=1` hides physical GPU 0, PyTorch correctly names physical GPU 1 as logical `cuda:0`. The assertions below prevent accidental multi-GPU execution and reject 4-bit loading.

In [ ]:
import gc
import hashlib
import json
import math
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import disable_progress_bars, load_dataset

disable_progress_bars()
assert os.environ.get("CUDA_VISIBLE_DEVICES") == "1", "GPU mask changed before torch import."
assert torch.cuda.is_available(), "A CUDA GPU is required."
assert torch.cuda.device_count() == 1, f"Expected exactly one visible GPU, got {torch.cuda.device_count()}."
torch.cuda.set_device(0)  # logical cuda:0 == physical GPU 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path(OUTPUT_DIR).resolve()
RAW_DIR = OUTPUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Visible GPU: logical cuda:0 -> {torch.cuda.get_device_name(0)}", flush=True)
print(f"Output directory: {OUTPUT_DIR}", flush=True)

from unsloth import FastLanguageModel

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Loading {MODEL_NAME} with Unsloth; dtype={DTYPE}; load_in_4bit=False", flush=True)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=False,
    device_map={"": 0},
)
FastLanguageModel.for_inference(model)
model.eval()

assert not bool(getattr(model, "is_loaded_in_4bit", False)), "Unexpected 4-bit model."
quant_method = getattr(getattr(model, "config", None), "quantization_config", None)
assert quant_method is None, f"Unexpected quantization config: {quant_method}"
parameter_devices = {str(parameter.device) for parameter in model.parameters()}
assert parameter_devices == {"cuda:0"}, f"Model is not fully confined to logical cuda:0: {parameter_devices}"
floating_dtypes = {str(parameter.dtype) for parameter in model.parameters() if parameter.is_floating_point()}
print(f"Model floating dtypes: {sorted(floating_dtypes)}", flush=True)
print(f"CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)

## Official chat-template audit

The model repository defines a Qwen ChatML Jinja template. With `add_generation_prompt=True`, a one-turn prompt must end in `<|im_start|>assistant\n<think>\n`. This is a thinking-only model, so this notebook deliberately does **not** pass `enable_thinking`.

In [ ]:
assert tokenizer.chat_template, "The tokenizer does not contain a chat template."
template_sha256 = hashlib.sha256(tokenizer.chat_template.encode("utf-8")).hexdigest()
audit_messages = [{"role": "user", "content": "What is 1 + 1?"}]
audit_rendered = tokenizer.apply_chat_template(
    audit_messages, tokenize=False, add_generation_prompt=True
)
expected_suffix = "<|im_start|>assistant\n<think>\n"
assert audit_rendered.endswith(expected_suffix), (
    "Unexpected Qwen3-4B-Thinking-2507 template ending. Expected "
    + repr(expected_suffix) + " but got " + repr(audit_rendered[-100:])
)
template_audit = {
    "model_name": MODEL_NAME,
    "tokenizer_class": tokenizer.__class__.__name__,
    "eos_token": tokenizer.eos_token,
    "eos_token_id": tokenizer.eos_token_id,
    "chat_template_sha256": template_sha256,
    "rendered_example": audit_rendered,
    "expected_generation_suffix": expected_suffix,
    "uses_enable_thinking_argument": False,
    "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    "official_model_card": "https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507",
    "official_tokenizer_config": "https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507/blob/main/tokenizer_config.json",
}
(OUTPUT_DIR / "template_audit.json").write_text(
    json.dumps(template_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"chat_template SHA256: {template_sha256}", flush=True)
print("Rendered audit prompt:\n" + audit_rendered, flush=True)

## Shared generation and persistence helpers

Each completed sample is appended immediately to JSONL and flushed to disk. A run fingerprint prevents resume mode from silently mixing different models, prompts, or decoding settings. No `tqdm` progress bar is used.

In [ ]:
PROMPT_VERSION = "student-base-eval-v2-unsloth-seeding"
fingerprint_payload = {
    "model": MODEL_NAME,
    "prompt_version": PROMPT_VERSION,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "seed": SEED,
    "template_sha256": template_sha256,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Run fingerprint: {RUN_FINGERPRINT}", flush=True)

def format_seconds(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def print_progress(label, done, total, started_at):
    percent = 100.0 * done / total if total else 100.0
    print(
        f"[PROGRESS] {label}: {done}/{total} ({percent:.2f}%) "
        f"elapsed={format_seconds(time.time() - started_at)}",
        flush=True,
    )

def stable_seed(sample_id):
    digest = hashlib.sha256(str(sample_id).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)

def split_thinking(generated_text):
    text = generated_text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False

def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def load_matching_records(path):
    latest = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated JSONL line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == RUN_FINGERPRINT:
                latest[str(record["sample_id"])] = record
    return latest

def limit_dataset(dataset, limit):
    if limit is None:
        return dataset
    return dataset.select(range(min(int(limit), len(dataset))))

def generate_one(user_prompt, max_new_tokens, sample_id):
    messages = [{"role": "user", "content": user_prompt}]
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    assert rendered.endswith(expected_suffix)
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Prompt has {input_tokens} tokens and exceeds MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}.")
    effective_max_new_tokens = min(int(max_new_tokens), int(available))
    if effective_max_new_tokens < max_new_tokens:
        print(
            f"[WARN] {sample_id}: capped max_new_tokens from {max_new_tokens} "
            f"to {effective_max_new_tokens} to fit the context window.",
            flush=True,
        )
    # Unsloth's patched generate() does not accept the Transformers `generator` kwarg.
    # Reset all global RNGs per sample instead, preserving deterministic resume behavior.
    sample_seed = stable_seed(sample_id)
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=effective_max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_thinking(raw_generation)
    return {
        "sample_seed": sample_seed,
        "raw_generation": raw_generation,
        "thinking": thinking,
        "final_answer": final_answer,
        "found_think_close": found_think_close,
        "input_tokens": input_tokens,
        "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": effective_max_new_tokens,
        "generation_seconds": time.time() - started,
    }

In [ ]:
# Dataset loading and schema checks. IFBench/IFEval call their sole evaluation split 'train'.
datasets_to_run = {}
if RUN_MATH:
    datasets_to_run["aime25"] = limit_dataset(load_dataset("math-ai/aime25", split="test"), LIMIT_AIME25)
    datasets_to_run["aime26"] = limit_dataset(load_dataset("math-ai/aime26", split="test"), LIMIT_AIME26)
    for name in ("aime25", "aime26"):
        required = {"problem", "answer"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_IF:
    datasets_to_run["ifbench"] = limit_dataset(load_dataset("allenai/IFBench_test", split="train"), LIMIT_IFBENCH)
    datasets_to_run["ifeval"] = limit_dataset(load_dataset("google/IFEval", split="train"), LIMIT_IFEVAL)
    for name in ("ifbench", "ifeval"):
        required = {"key", "prompt", "instruction_id_list", "kwargs"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_SWE:
    datasets_to_run["swe_verified"] = limit_dataset(
        load_dataset("princeton-nlp/SWE-bench_Verified", split="test"), LIMIT_SWE
    )
    required = {"instance_id", "repo", "base_commit", "problem_statement"}
    assert required.issubset(datasets_to_run["swe_verified"].column_names), datasets_to_run["swe_verified"].column_names

dataset_audit = {
    name: {"rows": len(dataset), "columns": dataset.column_names}
    for name, dataset in datasets_to_run.items()
}
(OUTPUT_DIR / "dataset_audit.json").write_text(
    json.dumps(dataset_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(dataset_audit, ensure_ascii=False, indent=2), flush=True)

## Math: AIME25 and AIME26

The primary metric is exact match after extracting the last `\boxed{...}` (with conservative fallbacks) and normalizing the AIME integer answer.

In [ ]:
def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index]
    return None

def extract_math_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches:
        return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""

def normalize_math_answer(value):
    value = str(value).strip()
    value = value.replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value

def run_math_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    started = time.time()
    if completed:
        print(f"[RESUME] student-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        if sample_id in completed:
            done += 1
            continue
        prompt = (
            str(example["problem"]).strip()
            + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
        )
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "problem": example["problem"],
            "gold_answer": str(example["answer"]),
        }
        try:
            result = generate_one(prompt, MAX_NEW_TOKENS, f"{name}:{sample_id}")
            predicted = extract_math_answer(result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "predicted_answer": predicted,
                "correct": normalize_math_answer(predicted) == normalize_math_answer(example["answer"]),
            }
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "correct": False}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

math_records = {}
if RUN_MATH:
    math_records["aime25"] = run_math_benchmark("aime25", datasets_to_run["aime25"])
    math_records["aime26"] = run_math_benchmark("aime26", datasets_to_run["aime26"])

## Instruction following: IFBench and IFEval

The code below mirrors AllenAI's official strict/loose verifier logic. Crucially, only text after the model's final `</think>` is checked; hidden reasoning must not contaminate length, keyword, or formatting constraints. Primary reported scores are prompt-level loose accuracy, alongside all four official aggregate variants.

In [ ]:
from ifbench import instructions_registry

def response_variants(response):
    lines = response.split("\n")
    remove_first = "\n".join(lines[1:]).strip()
    remove_last = "\n".join(lines[:-1]).strip()
    remove_both = "\n".join(lines[1:-1]).strip()
    return [
        response,
        response.replace("*", ""),
        remove_first,
        remove_last,
        remove_both,
        remove_first.replace("*", ""),
        remove_last.replace("*", ""),
        remove_both.replace("*", ""),
    ]

def check_instruction(instruction_id, kwargs, prompt, candidates):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)
    clean_kwargs = {key: value for key, value in dict(kwargs).items() if value is not None}
    instruction.build_description(**clean_kwargs)
    args = instruction.get_instruction_args()
    if args and "prompt" in args:
        instruction.build_description(prompt=prompt)
    return any(candidate.strip() and instruction.check_following(candidate) for candidate in candidates)

def score_instruction_response(example, response):
    strict = []
    loose = []
    for instruction_id, kwargs in zip(example["instruction_id_list"], example["kwargs"]):
        strict.append(check_instruction(instruction_id, kwargs, example["prompt"], [response]))
        loose.append(check_instruction(instruction_id, kwargs, example["prompt"], response_variants(response)))
    return strict, loose

def run_if_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    started = time.time()
    if completed:
        print(f"[RESUME] student-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("key", index))
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "prompt": example["prompt"],
            "instruction_id_list": example["instruction_id_list"],
            "kwargs": example["kwargs"],
        }
        try:
            result = generate_one(example["prompt"], MAX_NEW_TOKENS, f"{name}:{sample_id}")
            strict, loose = score_instruction_response(example, result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "strict_instruction_results": strict,
                "loose_instruction_results": loose,
                "strict_prompt_pass": all(strict),
                "loose_prompt_pass": all(loose),
            }
            consecutive_errors = 0
        except Exception as exc:
            consecutive_errors += 1
            failed = [False] * len(example["instruction_id_list"])
            record = {
                **base, "status": "error", "error": repr(exc),
                "strict_instruction_results": failed,
                "loose_instruction_results": failed,
                "strict_prompt_pass": False, "loose_prompt_pass": False,
            }
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation/scoring errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

if_records = {}
if RUN_IF:
    if_records["ifbench"] = run_if_benchmark("ifbench", datasets_to_run["ifbench"])
    if_records["ifeval"] = run_if_benchmark("ifeval", datasets_to_run["ifeval"])

## SWE-bench Verified

This base-model pass is a direct prompt-to-patch baseline: the model does not browse or edit a checked-out repository. Generated patches are exported in SWE-bench prediction format. When enabled, the official Docker harness is then invoked; only its resolved count becomes the reported score. Docker, substantial disk space, and the `swebench` package are required for that stage.

In [ ]:
def extract_patch(final_answer):
    fenced = re.findall(r"```(?:diff|patch)?\s*\n(.*?)```", final_answer, flags=re.DOTALL | re.IGNORECASE)
    candidates = fenced if fenced else [final_answer]
    for candidate in reversed(candidates):
        position = candidate.find("diff --git ")
        if position >= 0:
            return candidate[position:].strip()
    return candidates[-1].strip()

def make_swe_prompt(example):
    return f"""You are fixing a real software repository issue. Produce the patch that resolves the issue.
Repository: {example['repo']}
Base commit: {example['base_commit']}

Issue:
{example['problem_statement']}

Return the final answer as a valid unified git diff beginning with `diff --git`. Do not include prose outside the patch in the final answer."""

def run_swe_generation(dataset):
    name = "swe_verified"
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    started = time.time()
    if completed:
        print(f"[RESUME] student-base/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"student-base/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for example in dataset:
        sample_id = str(example["instance_id"])
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "repo": example["repo"],
            "base_commit": example["base_commit"],
            "problem_statement": example["problem_statement"],
        }
        try:
            result = generate_one(make_swe_prompt(example), MAX_NEW_TOKENS, f"{name}:{sample_id}")
            record = {**base, **result, "status": "ok", "model_patch": extract_patch(result["final_answer"])}
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "model_patch": ""}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"student-base/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

swe_records = {}
swe_predictions_path = RAW_DIR / "swe_predictions.jsonl"
if RUN_SWE:
    swe_records = run_swe_generation(datasets_to_run["swe_verified"])
    with swe_predictions_path.open("w", encoding="utf-8") as handle:
        for example in datasets_to_run["swe_verified"]:
            instance_id = str(example["instance_id"])
            record = swe_records.get(instance_id, {})
            prediction = {
                "instance_id": instance_id,
                "model_name_or_path": "student-base-Qwen3-4B-Thinking-2507",
                "model_patch": record.get("model_patch", ""),
            }
            handle.write(json.dumps(prediction, ensure_ascii=False) + "\n")
    print(f"SWE-bench predictions: {swe_predictions_path}", flush=True)

In [ ]:
# Invoke the official SWE-bench Docker harness and discover its JSON report.
# A harness failure leaves the score as N/A; it is never converted to zero or a proxy metric.
swe_harness = {"status": "not_run", "resolved": None, "total": None, "score": None}
if RUN_SWE and RUN_SWE_HARNESS:
    import subprocess
    command = [
        sys.executable, "-m", "swebench.harness.run_evaluation",
        "--dataset_name", "princeton-nlp/SWE-bench_Verified",
        "--predictions_path", str(swe_predictions_path.resolve()),
        "--max_workers", str(SWE_MAX_WORKERS),
        "--run_id", SWE_RUN_ID,
    ]
    harness_log = OUTPUT_DIR / "swe_harness.log"
    print("[SWE-HARNESS] " + " ".join(command), flush=True)
    try:
        harness_started_at = time.time()
        with harness_log.open("a", encoding="utf-8") as log_handle:
            process = subprocess.Popen(
                command, cwd=OUTPUT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1,
            )
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                print("[SWE-HARNESS] " + line.rstrip(), flush=True)
            return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(f"SWE-bench harness exited with status {return_code}; see {harness_log}")
        reports = []
        for report_path in OUTPUT_DIR.rglob("*.json"):
            try:
                report = json.loads(report_path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if (
                report_path.stat().st_mtime >= harness_started_at - 2
                and isinstance(report, dict)
                and ("resolved_ids" in report or "resolved_instances" in report)
            ):
                reports.append((report_path.stat().st_mtime, report_path, report))
        if not reports:
            raise RuntimeError("Harness completed but no result JSON containing resolved IDs/counts was found.")
        _, report_path, report = max(reports, key=lambda item: item[0])
        resolved_value = report.get("resolved_ids", report.get("resolved_instances"))
        resolved = len(resolved_value) if isinstance(resolved_value, list) else int(resolved_value)
        total_value = report.get("total_instances", len(datasets_to_run["swe_verified"]))
        total = len(total_value) if isinstance(total_value, list) else int(total_value)
        swe_harness = {
            "status": "complete", "resolved": resolved, "total": total,
            "score": resolved / total if total else None, "report_path": str(report_path),
        }
    except Exception as exc:
        swe_harness = {
            "status": "pending_harness", "resolved": None, "total": len(datasets_to_run["swe_verified"]),
            "score": None, "error": repr(exc),
        }
        print(f"[WARN] Official SWE-bench score unavailable: {exc!r}", flush=True)
elif RUN_SWE:
    swe_harness = {
        "status": "pending_harness", "resolved": None, "total": len(datasets_to_run["swe_verified"]),
        "score": None, "note": "Predictions generated; RUN_SWE_HARNESS=False.",
    }
(OUTPUT_DIR / "swe_harness_status.json").write_text(
    json.dumps(swe_harness, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(swe_harness, ensure_ascii=False, indent=2), flush=True)

## Aggregate metrics and bar chart

In [ ]:
metric_rows = []

def add_metric(benchmark, domain, metric, numerator, denominator, status="complete", note=""):
    score = numerator / denominator if numerator is not None and denominator else None
    metric_rows.append({
        "benchmark": benchmark, "domain": domain, "metric": metric,
        "score": score, "score_percent": 100 * score if score is not None else None,
        "numerator": numerator, "denominator": denominator, "status": status, "note": note,
    })

for name, records_by_id in math_records.items():
    selected_ids = [str(row.get("id", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    add_metric(name.upper(), "Math", "exact_match", sum(bool(record.get("correct")) for record in records), len(records))

for name, records_by_id in if_records.items():
    selected_ids = [str(row.get("key", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    strict_lists = [record.get("strict_instruction_results", []) for record in records]
    loose_lists = [record.get("loose_instruction_results", []) for record in records]
    instruction_total = sum(len(values) for values in strict_lists)
    add_metric(name.upper(), "Instruction Following", "prompt_level_strict", sum(bool(values) and all(values) for values in strict_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_strict", sum(sum(values) for values in strict_lists), instruction_total)
    add_metric(name.upper(), "Instruction Following", "prompt_level_loose", sum(bool(values) and all(values) for values in loose_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_loose", sum(sum(values) for values in loose_lists), instruction_total)

if RUN_SWE:
    if swe_harness.get("status") == "complete":
        add_metric("SWE-bench Verified", "SWE", "resolved_rate", swe_harness["resolved"], swe_harness["total"])
    else:
        metric_rows.append({
            "benchmark": "SWE-bench Verified", "domain": "SWE", "metric": "resolved_rate",
            "score": None, "score_percent": None, "numerator": None,
            "denominator": swe_harness.get("total"), "status": "pending_harness",
            "note": "Official Docker harness did not produce a resolved score.",
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_csv = OUTPUT_DIR / "metrics.csv"
metrics_json = OUTPUT_DIR / "metrics.json"
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
metrics_payload = {
    "model": MODEL_NAME, "run_fingerprint": RUN_FINGERPRINT,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": metric_rows, "swe_harness": swe_harness,
}
metrics_json.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")
display(metrics_df)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

primary_specs = [
    ("AIME25", "exact_match", "AIME25\nExact match"),
    ("AIME26", "exact_match", "AIME26\nExact match"),
    ("IFBENCH", "prompt_level_loose", "IFBench\nPrompt loose"),
    ("IFEVAL", "prompt_level_loose", "IFEval\nPrompt loose"),
    ("SWE-bench Verified", "resolved_rate", "SWE-bench Verified\nResolved"),
]
labels, values, available = [], [], []
for benchmark, metric, label in primary_specs:
    row = metrics_df[(metrics_df["benchmark"] == benchmark) & (metrics_df["metric"] == metric)]
    if row.empty:
        continue
    value = row.iloc[0]["score_percent"]
    labels.append(label)
    available.append(pd.notna(value))
    values.append(float(value) if pd.notna(value) else 0.0)

colors = ["#4C78A8", "#4C78A8", "#F58518", "#F58518", "#54A24B"][:len(labels)]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="#333333", linewidth=0.7)
for bar, value, is_available in zip(bars, values, available):
    if is_available:
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1.2, f"{value:.2f}%", ha="center", va="bottom", fontsize=10)
    else:
        bar.set_facecolor("#D9D9D9")
        bar.set_hatch("//")
        ax.text(bar.get_x() + bar.get_width() / 2, 2.0, "N/A\n(harness pending)", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(labels)), labels)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 108)
ax.set_title("Qwen3-4B-Thinking-2507: Base Student Evaluation")
ax.grid(axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()
figure_path = OUTPUT_DIR / "student_base_scores.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Metrics JSON: {metrics_json}", flush=True)
print(f"Metrics CSV:  {metrics_csv}", flush=True)
print(f"Bar chart:    {figure_path}", flush=True)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("[DONE] student-base evaluation notebook completed.", flush=True)
if RUN_SWE and swe_harness.get("status") != "complete":
    print("[ACTION REQUIRED] SWE-bench Verified remains N/A until the official Docker harness completes.", flush=True)